# Smart Recruitment Assistant — AI Screening System

A reproducible recruitment screening workflow. Target labels: `0` = Not Looking for Job Change, `1` = Looking for Job Change.

## Section 0 — Setup

In [ ]:
from pathlib import Path
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
APP_DIR = PROJECT_ROOT / "app"
sys.path.insert(0, str(APP_DIR))
from modeling import train_and_export

print(f"Project root: {PROJECT_ROOT}")

## 2. Load the Training Dataset

Load the training dataset that contains the target variable.

In [ ]:
## Section 1 — Data Understanding

DATA_PATH = PROJECT_ROOT / "data" / "aug_train.csv"
df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
display(df.head())
df.info()
display(df.describe(include="all").T)
display(df.isna().sum().sort_values(ascending=False).head(10))

## 3. Initial Data Inspection

Inspect the dataset structure, data types, statistical summary, and missing values before starting the preprocessing steps.

In [ ]:
# Dataset information
df.info()

In [ ]:
# Statistical summary of numerical features
df.describe()

In [ ]:
# Check missing values in each column
missing_values = df.isnull().sum()

print(missing_values[missing_values > 0])

## 4. Separate Features and Target

Separate the input features from the target variable.

In [ ]:
# Target column
target_column = "target"

# Separate features and target
X = df.drop(columns=[target_column]).copy()
y = df[target_column].copy()

print("Features shape:", X.shape)
print("Target shape:", y.shape)

## 5. Explore Ordinal Categorical Values

Inspect the unique values in `education_level` and `company_size` to identify their categories and determine the appropriate logical ordering before applying ordinal encoding.

In [ ]:
education_levels = df['education_level'].unique()
company_sizes = df['company_size'].unique()

print(f'Education Levels : {education_levels}')
print(f'Company Sizes : {company_sizes}')

## 6. Convert Experience to Numeric

Convert the `experience` feature from text categories into numeric years of experience.

In [ ]:
# Convert experience values into numeric years
def convert_experience(value):
    if pd.isna(value):
        return np.nan
    elif value == ">20":
        return 21
    elif value == "<1":
        return 0
    else:
        return float(value)

X["experience"] = X["experience"].apply(convert_experience)

display(X[["experience"]].head(10))

## 7. Train/Test Split

Split the data into 80% training and 20% testing while preserving the target class distribution.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

## 8. Define Categorical and Numerical Features

Identify the categorical and numerical features before handling missing values.

In [ ]:
# Numerical features
numerical_features = [
    "city_development_index",
    "training_hours",
    "experience"
]

# Ordinal categorical features
ordinal_features = [
    "education_level",
    "company_size",
    "last_new_job"
]

# Nominal categorical features
nominal_features = [
    "gender",
    "relevent_experience",
    "enrolled_university",
    "major_discipline",
    "company_type"
]

# City will be handled separately using frequency encoding
city_feature = ["city"]

print("Numerical features:")
print(numerical_features)

print("\nOrdinal features:")
print(ordinal_features)

print("\nNominal features:")
print(nominal_features)

print("\nCity feature:")
print(city_feature)

## 9. Handle Missing Numerical Values

Fill missing numerical values using the median calculated from the training data.

In [ ]:
# Create numerical imputer
numerical_imputer = SimpleImputer(strategy="median")

# Fit on training data and transform training data
X_train[numerical_features] = numerical_imputer.fit_transform(
    X_train[numerical_features]
)

# Apply the same transformation to test data
X_test[numerical_features] = numerical_imputer.transform(
    X_test[numerical_features]
)

## 10. Handle Missing Categorical Values

Replace missing categorical values with `Unknown`.

In [ ]:
# All categorical columns that need categorical imputation
categorical_features = ordinal_features + nominal_features + city_feature

# Create categorical imputer
categorical_imputer = SimpleImputer(
    strategy="constant",
    fill_value="Unknown"
)

# Fit on training data
X_train[categorical_features] = categorical_imputer.fit_transform(
    X_train[categorical_features]
)

# Apply the same transformation to test data
X_test[categorical_features] = categorical_imputer.transform(
    X_test[categorical_features]
)

print("Missing values in X_train:", X_train.isna().sum().sum())
print("Missing values in X_test:", X_test.isna().sum().sum())

In [ ]:
X_train.columns.tolist()

In [ ]:
eda_df = X_train.copy()
eda_df['target'] = y_train.values

print("Shape:", eda_df.shape)
eda_df.head()

In [ ]:
print(eda_df['target'].value_counts())
print()
print(eda_df['target'].value_counts(normalize=True) * 100)

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.figure(figsize=(6, 4))
sns.countplot(x='target', data=eda_df)
plt.title('Target Variable Distribution')
plt.xlabel('Target (0 = Not Looking, 1 = Looking for Job Change)')
plt.ylabel('Count')
plt.show()

In [ ]:
categorical_cols = ['gender', 'relevent_experience', 'enrolled_university',
                     'education_level', 'major_discipline', 'company_type']

for col in categorical_cols:
    plt.figure(figsize=(9, 5))
    sns.countplot(x=col, hue='target', data=eda_df)
    plt.title(f'{col} vs Target')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

In [ ]:
numerical_cols = eda_df.select_dtypes(include=['int64', 'float64']).columns

corr_matrix = eda_df[numerical_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap - Numerical Features')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
eda_df['education_level'].value_counts().plot(kind='bar', color='steelblue')
plt.title('Education Level Distribution')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
eda_df['experience'].value_counts().sort_index().plot(kind='bar', color='seagreen')
plt.title('Experience Distribution')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
sns.histplot(eda_df['training_hours'], bins=30, kde=True, color='darkorange')
plt.title('Training Hours Distribution')
plt.xlabel('Training Hours')
plt.show()



- الـ Target متوازن؟ لا، الداتا Imbalanced: 75% من المتقدمين مش بيدوروا على تغيير وظيفة، و25% بس بيدوروا (Target=1).
- أكثر Features تأثيرًا: عدم وجود "relevant experience" ونوع الشركة "Pvt Ltd" مرتبطين بزيادة احتمالية البحث عن وظيفة جديدة.
- في الـ Correlation Heatmap، فيه ارتباط عكسي متوسط (-0.34) بين city_development_index والـ target، يعني كل ما مؤشر تطور المدينة أعلى كل ما نسبة البحث عن وظيفة تقل.
- أكثر صفة شيوعًا بين المتقدمين: تعليم Graduate، تخصص STEM، ويشتغلوا في شركات Pvt Ltd، وأغلبهم عندهم ساعات تدريب قليلة نسبيًا.

## 11. Ordinal Encoding

Convert ordinal categorical features into numerical values while preserving their logical order.

In [ ]:
# Define logical order for education level
education_mapping = {
    "Unknown": -1,
    "Primary School": 0,
    "High School": 1,
    "Graduate": 2,
    "Masters": 3,
    "Phd": 4
}

# Define logical order for company size
company_size_mapping = {
    "Unknown": -1,
    "<10": 0,
    "10/49": 1,
    "50-99": 2,
    "100-500": 3,
    "500-999": 4,
    "1000-4999": 5,
    "5000-9999": 6,
    "10000+": 7
}

# Define logical order for last new job
last_new_job_mapping = {
    "Unknown": -1,
    "never": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    ">4": 5
}

# Apply mappings
X_train["education_level"] = X_train["education_level"].map(
    education_mapping
)

X_test["education_level"] = X_test["education_level"].map(
    education_mapping
)

X_train["company_size"] = X_train["company_size"].map(
    company_size_mapping
)

X_test["company_size"] = X_test["company_size"].map(
    company_size_mapping
)

X_train["last_new_job"] = X_train["last_new_job"].map(
    last_new_job_mapping
)

X_test["last_new_job"] = X_test["last_new_job"].map(
    last_new_job_mapping
)

# Quick check: confirm no NaN reappeared after mapping
print("NaN after ordinal mapping - X_train:", X_train[["education_level", "company_size", "last_new_job"]].isna().sum().sum())
print("NaN after ordinal mapping - X_test:", X_test[["education_level", "company_size", "last_new_job"]].isna().sum().sum())


## 12. Frequency Encoding for City

Convert the high-cardinality `city` feature into one numerical feature based on the frequency of each city in the training data.

In [ ]:
# Calculate city frequency
city_frequency = X_train["city"].value_counts(normalize=True)

# Map frequencies to training and testing data
X_train["city_freq"] = X_train["city"].map(city_frequency)
X_test["city_freq"] = X_test["city"].map(city_frequency)

# Unseen cities in test data get frequency 0
X_test["city_freq"] = X_test["city_freq"].fillna(0)

# Remove the original city column
X_train = X_train.drop(columns=["city"])
X_test = X_test.drop(columns=["city"])

print("City frequency encoding completed.")

## 13. Prepare Numerical Features for Scaling

Collect all numerical and ordinal features that should be scaled.

In [ ]:
# Numerical features selected for scaling
scaling_features = [
    "city_development_index",
    "training_hours",
    "experience",
    "city_freq"
]

print("Features to scale:")
print(scaling_features)

## 14. One-Hot Encode Nominal Features

Convert nominal categorical features into numerical binary features.

In [ ]:
# Create One-Hot Encoder
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

# Fit encoder on training data only
X_train_encoded = encoder.fit_transform(
    X_train[nominal_features]
)

# Transform test data using the same encoder
X_test_encoded = encoder.transform(
    X_test[nominal_features]
)

# Get feature names
encoded_feature_names = encoder.get_feature_names_out(
    nominal_features
)

# Convert encoded arrays to DataFrames
X_train_encoded = pd.DataFrame(
    X_train_encoded,
    columns=encoded_feature_names,
    index=X_train.index
)

X_test_encoded = pd.DataFrame(
    X_test_encoded,
    columns=encoded_feature_names,
    index=X_test.index
)

print("Encoded train shape:", X_train_encoded.shape)
print("Encoded test shape:", X_test_encoded.shape)

## 15. Combine Processed Features

Combine numerical, ordinal, frequency-encoded, and One-Hot encoded features into the final feature matrices.

In [ ]:
# Features that remain as numerical/ordinal columns
base_features = [
    "city_development_index",
    "training_hours",
    "experience",
    "education_level",
    "company_size",
    "last_new_job",
    "city_freq"
]

# Select base features
X_train_base = X_train[base_features].copy()
X_test_base = X_test[base_features].copy()

# Combine with One-Hot encoded features
X_train_processed = pd.concat(
    [X_train_base, X_train_encoded],
    axis=1
)

X_test_processed = pd.concat(
    [X_test_base, X_test_encoded],
    axis=1
)

print("Processed X_train shape:", X_train_processed.shape)
print("Processed X_test shape:", X_test_processed.shape)

## 16. Feature Scaling

Apply StandardScaler to the selected numerical features using statistics learned from the training data only.

In [ ]:
# Create StandardScaler
scaler = StandardScaler()

# Fit scaler on training data
X_train_processed[scaling_features] = scaler.fit_transform(
    X_train_processed[scaling_features]
)

# Apply the same scaler to test data
X_test_processed[scaling_features] = scaler.transform(
    X_test_processed[scaling_features]
)

print("Feature scaling completed.")

## 17. Create Final Train/Test Variables

Create the final variables that will be passed to the next stage of the project.

In [ ]:
# Final processed feature matrices
X_train = X_train_processed.copy()
X_test = X_test_processed.copy()

# Reset indexes
X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)

y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

print("Final variables created successfully.")

## 18. Validate the Final Outputs

Check that the final datasets contain no missing values and that the train/test target distributions remain balanced.

In [ ]:
# Check missing values
train_missing = X_train.isna().sum().sum()
test_missing = X_test.isna().sum().sum()

print("Missing values in X_train:", train_missing)
print("Missing values in X_test:", test_missing)

# Check shapes
print("\nFinal shapes:")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

# Check feature consistency
print(
    "\nSame feature columns:",
    list(X_train.columns) == list(X_test.columns)
)

# Check target distribution
print("\nTarget distribution - Train:")
print(y_train.value_counts(normalize=True))

print("\nTarget distribution - Test:")
print(y_test.value_counts(normalize=True))

# Final validation
assert train_missing == 0
assert test_missing == 0
assert len(X_train) == len(y_train)
assert len(X_test) == len(y_test)
assert list(X_train.columns) == list(X_test.columns)

print("\nData Cleaning & Preprocessing completed successfully!")
print("Data is ready for modeling.")

## 19.Creat Logistic Regression Model
Create the Logistic Regression model object that will be used for training and prediction.

In [ ]:
from sklearn.linear_model import LogisticRegression

##20.Create The Model Object

In [ ]:
model = LogisticRegression(max_iter=1000)

##21. Train The Model
Train the Logistic Regression model using the training data

In [ ]:
model.fit(X_train, y_train)

##22. Make Predictions
Make predictions on the test data using the trained model.

In [ ]:
y_pred=model.predict(X_test)

## Section 4 — Logistic Regression

Evaluate Accuracy, Precision, Recall, F1, Balanced Accuracy, ROC-AUC, confusion matrix, classification report, and coefficient direction. Positive coefficients increase estimated job-change likelihood; negative coefficients decrease it.

In [ ]:
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)



##24. Confusion Matrix
Generate the confusion matrix to analyze the classification results.

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

##25.Confusion Matrix
Visualize the confusion matrix

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Positive", "Negative"],
    yticklabels=["Positive", "Negative"]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Logistic Regression - Confusion Matrix")
plt.show()

##26. Save the Model
Save the trained Logistic Regression model for later use.

In [ ]:
import joblib
import os

# Create the folder automatically if it doesn't exist
os.makedirs("models", exist_ok=True)

# save the model
joblib.dump(model, "models/logistic_regression.pkl")

print("Logistic Regression model saved successfully.")

##27. Create Random Forest Model

In [ ]:
from sklearn.ensemble import RandomForestClassifier
# Create the Random Forest model
rf_model=RandomForestClassifier(n_estimators=200,
                                random_state=42,
                                class_weight="balanced",n_jobs=-1)

##28. Train Random Forest

In [ ]:
# Train the Random Forest using the same training data
rf_model.fit(X_train,y_train)

##29. Make Random Forest Predictions

In [ ]:
# Make predictions on the test set
rf_pred=rf_model.predict(X_test)

##30. Evaluate Random Forest Performance

In [ ]:
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score

# Calculate evaluation metrics
rf_accuracy = accuracy_score(y_test, rf_pred)
rf_precision = precision_score(y_test, rf_pred)
rf_recall = recall_score(y_test, rf_pred)
rf_f1 = f1_score(y_test, rf_pred)

print("Random Forest Performance")
print("-------------------------")
print("Accuracy :", rf_accuracy)
print("Precision:", rf_precision)
print("Recall   :", rf_recall)
print("F1 Score :", rf_f1)

## 31. Random Forest Classification Report

In [ ]:
from sklearn.metrics import classification_report

print("Random Forest Classification Report:")
print(classification_report(y_test, rf_pred))

##32. Random Forest Confusion Matrix

In [ ]:
rf_cm = confusion_matrix(y_test, rf_pred)

print("Random Forest Confusion Matrix:")
print(rf_cm)



## 33. Visualize Random Forest Confusion Matrix

In [ ]:
plt.figure(figsize=(6, 4))
sns.heatmap(
    rf_cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Predicted 0", "Predicted 1"],
    yticklabels=["Actual 0", "Actual 1"]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Random Forest Confusion Matrix")
plt.show()

##34. Random Forest Feature Importance

In [ ]:
# Extract feature importance from the trained Random Forest
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf_model.feature_importances_
})
# Sort from most important to least important
feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)
feature_importance

##35. Top 10 Important Features

In [ ]:
# Display the 10 most important features
feature_importance.head(10)

##36. Save Random Forest Model

In [ ]:
import os
import joblib

os.makedirs("models", exist_ok=True)

# Save the trained Random Forest model
joblib.dump(
    rf_model,
    "models/random_forest.pkl"
)

print("Random Forest model saved successfully.")

##37. Model Compariso

In [ ]:
# Compare both models using the same evaluation metrics
comparison = {
    "Model": ["Logistic Regression", "Random Forest"],
    "Accuracy": [accuracy,rf_accuracy],
    "Precision": [precision,rf_precision],
    "Recall": [recall,rf_recall],
    "F1 Score": [f1,rf_f1]
}

comparison_df = pd.DataFrame(comparison)

comparison_df

## 38. Visual Model Comparison

In [ ]:
comparison_plot = comparison_df.set_index("Model")

comparison_plot.plot(
    kind="bar",
    figsize=(10, 6)
)

plt.title("Model Comparison")
plt.ylabel("Score")
plt.xlabel("Model")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(title="Metrics")

plt.show()

## Section 5 — Random Forest and Model Comparison

The final selection is calculated from the exported metrics. Recall is the primary metric because missing a potentially interested candidate matters in screening; F1 is the tie-breaker. See `reports/model_comparison.csv` for the held-out results.

## 40. Top 10 Candidates by Confidence

In [ ]:
# Get the probability of Class 1
rf_probability = rf_model.predict_proba(X_test)[:, 1]

# Create a copy of the test data
top_candidates = X_test.copy()

# Add prediction probability
top_candidates["confidence"] = rf_probability

# Sort candidates by confidence
top_candidates = top_candidates.sort_values(
    by="confidence",
    ascending=False
)

# Display the top 10 candidates
top_candidates.head(10)

## Section 6 — Dashboard Reports and Streamlit Integration

The canonical training function uses the shared preprocessing implementation, compares Logistic Regression, Random Forest, and HistGradientBoosting on the same stratified split, selects the highest composite score across Recall, F1, Balanced Accuracy, and ROC-AUC, and exports the artifacts consumed by Streamlit.

In [ ]:
result = train_and_export(data_path=PROJECT_ROOT / "data" / "aug_train.csv", output_root=PROJECT_ROOT)
comparison_df = result["comparison"]
selected_model_name = result["selected_model"]
print("Selected model:", selected_model_name)
display(comparison_df)